In [ ]:
from pathlib import Path
import arviz as az

from autumn.projects.sm_covid2.common_school.calibration import get_bcm_object
from autumn.projects.sm_covid2.common_school.runner_tools import sample_with_pymc,extract_sample_subset, run_full_runs, get_uncertainty_dfs, calculate_diff_output_quantiles

In [2]:
iso3 = "IDN"
analysis = "main"
full_runs_samples = 1000

In [3]:
load_idata = True

if load_idata:
    idata = az.from_netcdf(
        Path.home() / "Models/AuTuMN_new/user/rragonnet/remote_run_outputs" / "33489767_test_full_analysis_24Jan2024_main" / "IDN" / "idata.nc"
    )
    burn_in = 20000
else:
    initvals =None
    bcm = get_bcm_object(iso3)
    
    idata = sample_with_pymc(bcm, initvals, draws=100, tune=50, cores=4, chains=4, method="DEMetropolisZ", sampler_options=None)
    burn_in = 10


### The following cell runs fine for both loaded and rerun idatas

In [ ]:
sampled_params = extract_sample_subset(idata, full_runs_samples, burn_in)  
full_runs = run_full_runs(sampled_params, iso3, analysis)

### This is where we have a problem. Working fine with rerun idata but not with loaded idata. 
The unc_dfs object is empty in the latter case.

In [5]:
import autumn.projects.sm_covid2.common_school.runner_tools as rt
from importlib import reload

In [6]:
reload(rt)
unc_dfs = rt.get_uncertainty_dfs(full_runs)

In [ ]:
Path.home()

In [11]:
out_path = Path.home() / "Models/AuTuMN_new/user/rragonnet/remote_run_outputs/IDN_deltaclosures_sa"

for scenario, unc_df in unc_dfs.items():
    unc_df.to_parquet(out_path / f"uncertainty_df_{scenario}.parquet")

In [13]:
diff_quantiles_df = calculate_diff_output_quantiles(full_runs)
diff_quantiles_df.to_parquet(out_path / "diff_quantiles_df.parquet")


# GENERATE MAIN COMPARISON OUTPUTS

In [14]:
import pandas as pd

In [16]:
basecase_unc_df = pd.read_parquet(
    Path.home() / "Models/AuTuMN_new/user/rragonnet/remote_run_outputs" / "33489767_test_full_analysis_24Jan2024_main" / "IDN" / "uncertainty_df_baseline.parquet"
)
delta_closed_unc_df =  pd.read_parquet(
    out_path / f"uncertainty_df_baseline.parquet"
)

In [ ]:
uncertainty_dfs = {
    'baseline': basecase_unc_df,
    'scenario_1': delta_closed_unc_df
}

In [ ]:
from matplotlib import pyplot as plt
import autumn.projects.sm_covid2.common_school.output_plots.country_highlight as ch
import autumn.projects.sm_covid2.common_school.output_plots.country_spec as cs
from importlib import reload

reload(ch)
reload(cs)
plt.rcParams["font.family"] = "Times New Roman"

for include_unc in [True, False]:
    fig, axes = plt.subplots(2,1,figsize=(10,6))

    for i, output in enumerate(["infection_deaths_ma7", "prop_ever_infected"]):
        sc_compare_ax = axes[i]
        ch._plot_two_scenarios(sc_compare_ax, uncertainty_dfs, output, iso3, include_unc=include_unc, include_legend=[True, False][i])

        ch.add_variant_emergence(sc_compare_ax, iso3)

        cs.format_date_axis(sc_compare_ax)
    # remove_axes_box(sc_compare_ax)
        ch.ad_panel_number(sc_compare_ax, ["A", "B"][i])

    suffix = "unc" if include_unc else "median"
    plt.savefig(f"delta_closures_sa_{suffix}.pdf")